# K Nearest Neighbor (KNN) from Scratch

In [ ]:
# Uncomment and run this cell to install the required packages.
# Note: If you are on a modern Linux/macOS and get an "externally-managed-environment" error, 
# please run this notebook inside a Python virtual environment (venv) or use the flag below:
# %pip install numpy pandas matplotlib seaborn scikit-learn --break-system-packages

# %pip install numpy pandas matplotlib seaborn scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

%matplotlib inline

iris = load_iris()
X = iris.data[:, :2]
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

## 1) Distance Metrics from Scratch

In [ ]:
def euclidean_distance(u, v):
    u, v = np.asarray(u, dtype=float), np.asarray(v, dtype=float)
    return np.sqrt(np.sum((u - v) ** 2))

def manhattan_distance(u, v):
    u, v = np.asarray(u, dtype=float), np.asarray(v, dtype=float)
    return np.sum(np.abs(u - v))

def cosine_distance(u, v):
    u, v = np.asarray(u, dtype=float), np.asarray(v, dtype=float)
    norm = np.linalg.norm(u) * np.linalg.norm(v)
    if norm == 0:
        return 1.0
    return 1.0 - np.dot(u, v) / norm

## 2) KNN Classifier from Scratch

In [ ]:
class KNNClassifierScratch:
    def __init__(self, k=3, metric='euclidean'):
        self.k = k
        self.metric = metric
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def _compute_dist(self, u, v):
        if self.metric == 'manhattan':
            return manhattan_distance(u, v)
        elif self.metric == 'cosine':
            return cosine_distance(u, v)
        else:
            return euclidean_distance(u, v)

    def _predict_single(self, x_q):
        distances = np.array([self._compute_dist(x_q, x_t) for x_t in self.X_train])
        nearest_idx = np.argsort(distances)[:self.k]
        nearest_labels = self.y_train[nearest_idx]
        return Counter(nearest_labels).most_common(1)[0][0]

    def predict(self, X):
        return np.array([self._predict_single(x) for x in X])

    def score(self, X, y):
        return np.mean(self.predict(X) == y)

## 3) Evaluation Metrics from Scratch

In [ ]:
def compute_accuracy(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return np.sum(y_true == y_pred) / len(y_true)

def compute_confusion_matrix(y_true, y_pred, num_classes=3):
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm

model = KNNClassifierScratch(k=3, metric='euclidean')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", compute_accuracy(y_test, y_pred))
print("Confusion Matrix:")
print(compute_confusion_matrix(y_test, y_pred, num_classes=3))

for m in ['euclidean', 'manhattan', 'cosine']:
    knn_m = KNNClassifierScratch(k=3, metric=m).fit(X_train, y_train)
    print(f"{m}: {knn_m.score(X_test, y_test):.4f}")

## 4) Decision Boundary Visualization

In [ ]:
def plot_decision_boundary(model, X, y, title, ax):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05),
                         np.arange(y_min, y_max, 0.05))
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.viridis)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.viridis, edgecolor='k', s=30)
    ax.set_title(title)
    ax.set_xlabel(iris.feature_names[0])
    ax.set_ylabel(iris.feature_names[1])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, k in zip(axes, [1, 5, 15]):
    knn_k = KNNClassifierScratch(k=k).fit(X_train, y_train)
    acc = knn_k.score(X_test, y_test)
    plot_decision_boundary(knn_k, X_train, y_train, f"K = {k} (test acc = {acc:.2f})", ax)
plt.tight_layout()
plt.show()